# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

# Read the private token from Colab Secrets.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN was not found. Add it through the Colab Secrets panel."
    )

# Connect DuckDB to the gated Hugging Face dataset.
con = duckdb.connect()

safe_token = HF_TOKEN.replace("'", "''")
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 will be used for the three verification checks.
MARCH_TABLE = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

# February provides the feature window.
# March provides the later outcome window.
FEB_MAR_TABLE = (
    "read_parquet(["
    f"'{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    "])"
)

print("Connected successfully to the FlyRank warehouse.")

Connected successfully to the FlyRank warehouse.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Data contract answer

One row in my final modelling frame represents one pseudonymised content page at one monthly decision point.

I will use February 2026 as the feature window and March 2026 as the later outcome window. Features describe information available before the decision, while the March outcome is used to determine whether the page showed declining performance afterward.

The final month contained in the `_sample` table will remain sealed and will not be used to design the label, select features, or improve the model.

In [2]:
# Inspect the available fields before defining the contract.
schema_df = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM {MARCH_TABLE}
    """
).df()

display(schema_df)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Data contract — five plain-language answers

1. **What does one row represent?**  
   One row in my final modelling frame represents one pseudonymised content page for one client at the end of February 2026.

2. **Which table will I use?**  
   I will use `fact_content_daily_performance` because it contains daily Google Search Console and Google Analytics performance for each pseudonymised content page.

3. **What time window will I use?**  
   February 2026 will be the feature window, and March 2026 will be the later outcome window. The final month in the `_sample` table will remain sealed.

4. **What will I predict or rank?**  
   I will predict whether a page's average daily Google Search clicks decline in March compared with February. The model probability will be used to rank pages for human review.

5. **What will I deliberately exclude?**  
   I will exclude March performance metrics and any field derived from the label from the feature set because they would not be known at the February decision point.

### Field roles

#### Features — maximum five

All features are calculated from February 2026:

1. `feb_avg_daily_impressions` — average daily Google Search impressions  
2. `feb_avg_daily_clicks` — average daily Google Search clicks  
3. `feb_avg_position` — impression-weighted average search position  
4. `feb_avg_daily_sessions` — average daily GA4 sessions  
5. `feb_engagement_rate` — engaged sessions divided by total sessions  

#### Label

`declined_next_month`

- `1` means March average daily clicks are lower than February average daily clicks.
- `0` means March average daily clicks are equal to or higher than February.

#### Context fields

- `client_hash_id`
- `content_hash_id`
- `decision_month`

These fields identify the row but will not be used as model features.

#### Excluded fields

- March impressions, clicks, position, sessions and engagement metrics
- Any feature calculated directly from `declined_next_month`
- Raw identifiers as model inputs
- The final sealed `_sample` month

These fields are excluded to prevent leakage and memorisation.

In [3]:
feature_fields = [
    "feb_avg_daily_impressions",
    "feb_avg_daily_clicks",
    "feb_avg_position",
    "feb_avg_daily_sessions",
    "feb_engagement_rate",
]

label_field = "declined_next_month"

context_fields = [
    "client_hash_id",
    "content_hash_id",
    "decision_month",
]

excluded_fields = [
    "march_avg_daily_impressions",
    "march_avg_daily_clicks",
    "march_avg_position",
    "march_avg_daily_sessions",
    "march_engagement_rate",
    "client_hash_id_as_feature",
    "content_hash_id_as_feature",
    "latest_sample_month",
]

print("Features:", feature_fields)
print("Label:", label_field)
print("Context:", context_fields)
print("Excluded:", excluded_fields)

Features: ['feb_avg_daily_impressions', 'feb_avg_daily_clicks', 'feb_avg_position', 'feb_avg_daily_sessions', 'feb_engagement_rate']
Label: declined_next_month
Context: ['client_hash_id', 'content_hash_id', 'decision_month']
Excluded: ['march_avg_daily_impressions', 'march_avg_daily_clicks', 'march_avg_position', 'march_avg_daily_sessions', 'march_engagement_rate', 'client_hash_id_as_feature', 'content_hash_id_as_feature', 'latest_sample_month']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain verification

This query checks whether each March row is uniquely identified by
`report_date`, `client_hash_id`, and `content_hash_id`.

In [4]:
grain_check = con.execute(
    f"""
    WITH base AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id
        FROM {MARCH_TABLE}
    ),
    unique_grain AS (
        SELECT DISTINCT
            report_date,
            client_hash_id,
            content_hash_id
        FROM base
    )
    SELECT
        (SELECT COUNT(*) FROM base) AS total_rows,
        (SELECT COUNT(*) FROM unique_grain) AS distinct_grain_rows,
        (SELECT COUNT(*) FROM base)
            - (SELECT COUNT(*) FROM unique_grain) AS duplicate_rows,
        (
            SELECT COUNT(*)
            FROM base
            WHERE report_date IS NULL
               OR client_hash_id IS NULL
               OR content_hash_id IS NULL
        ) AS null_key_rows
    """
).df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_grain_rows,duplicate_rows,null_key_rows
0,9841378,9841378,0,0


**Finding:** The total row count matched the number of distinct
date-client-content combinations, with no duplicate grain rows. This confirms
that one source row represents one content page for one client on one date.

### Query 2 — Slice size and date span

This query checks the number of rows, clients, content pages, and reporting dates
contained in the March 2026 slice.

In [6]:
slice_check = con.execute(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count,
        COUNT(DISTINCT report_date) AS reporting_days,
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date
    FROM {MARCH_TABLE}
    """
).df()

display(slice_check)

,row_count,client_count,content_count,reporting_days,first_report_date,last_report_date
0,9841378,55,331437,31,2026-03-01,2026-03-31


**Finding:** The March 2026 slice contains **[row count] rows** covering
**[first date] to [last date]**, with **[client count] clients** and
**[content count] content pages**.

### Query 3 — Availability check

This query uses explicit `IS TRUE` filters to count how many March rows have
GSC data, GA4 data, and both sources available.

In [7]:
availability_check = con.execute(
    f"""
    WITH
    total AS (
        SELECT COUNT(*) AS total_rows
        FROM {MARCH_TABLE}
    ),
    gsc_available AS (
        SELECT COUNT(*) AS gsc_rows
        FROM {MARCH_TABLE}
        WHERE gsc_data_available IS TRUE
    ),
    ga4_available AS (
        SELECT COUNT(*) AS ga4_rows
        FROM {MARCH_TABLE}
        WHERE ga4_data_available IS TRUE
    ),
    both_available AS (
        SELECT COUNT(*) AS both_rows
        FROM {MARCH_TABLE}
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    )
    SELECT
        total_rows,
        gsc_rows,
        ga4_rows,
        both_rows,
        ROUND(100.0 * both_rows / NULLIF(total_rows, 0), 2)
            AS both_available_percent
    FROM total
    CROSS JOIN gsc_available
    CROSS JOIN ga4_available
    CROSS JOIN both_available
    """
).df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_rows,ga4_rows,both_rows,both_available_percent
0,9841378,3611061,413966,364347,3.7


**Finding:** **[364347] rows**, or **[3.7]%** of the March slice,
have both GSC and GA4 data available. These rows are suitable for features that
combine search and analytics signals.

### Five-feature modelling frame

I aggregate the daily warehouse records into one row per pseudonymised
client-content page at the February 2026 decision point.

The five model features use only February information. March clicks are used
only to create the future label and are not included as model inputs.

In [8]:
# Build one modelling row per client-content page.
# February supplies the features; March supplies the future label.

model_frame = con.execute(
    f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            month,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_data_available IS TRUE THEN report_date
                END
            ) AS gsc_observed_days,

            COUNT(
                DISTINCT CASE
                    WHEN ga4_data_available IS TRUE THEN report_date
                END
            ) AS ga4_observed_days,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN gsc_data_available IS TRUE THEN report_date
                    END
                ),
                0
            ) AS avg_daily_impressions,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_clicks, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN gsc_data_available IS TRUE THEN report_date
                    END
                ),
                0
            ) AS avg_daily_clicks,

            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                SUM(
                    CASE
                        WHEN gsc_data_available IS TRUE
                        THEN COALESCE(gsc_impressions, 0)
                        ELSE 0
                    END
                ),
                0
            ) AS weighted_avg_position,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN COALESCE(ga4_sessions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                COUNT(
                    DISTINCT CASE
                        WHEN ga4_data_available IS TRUE THEN report_date
                    END
                ),
                0
            ) AS avg_daily_sessions,

            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE
                    THEN COALESCE(ga4_engaged_sessions, 0)
                    ELSE 0
                END
            )::DOUBLE
            / NULLIF(
                SUM(
                    CASE
                        WHEN ga4_data_available IS TRUE
                        THEN COALESCE(ga4_sessions, 0)
                        ELSE 0
                    END
                ),
                0
            ) AS engagement_rate

        FROM {FEB_MAR_TABLE}
        WHERE month IN ('2026-02', '2026-03')
        GROUP BY
            client_hash_id,
            content_hash_id,
            month
    ),

    february AS (
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_observed_days AS feb_gsc_days,
            ga4_observed_days AS feb_ga4_days,
            avg_daily_impressions AS feb_avg_daily_impressions,
            avg_daily_clicks AS feb_avg_daily_clicks,
            weighted_avg_position AS feb_avg_position,
            avg_daily_sessions AS feb_avg_daily_sessions,
            engagement_rate AS feb_engagement_rate
        FROM monthly
        WHERE month = '2026-02'
    ),

    march AS (
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_observed_days AS march_gsc_days,
            avg_daily_clicks AS march_avg_daily_clicks
        FROM monthly
        WHERE month = '2026-03'
    )

    SELECT
        feb.client_hash_id,
        feb.content_hash_id,
        '2026-02' AS decision_month,

        feb.feb_avg_daily_impressions,
        feb.feb_avg_daily_clicks,
        feb.feb_avg_position,
        feb.feb_avg_daily_sessions,
        feb.feb_engagement_rate,

        mar.march_avg_daily_clicks,

        CASE
            WHEN mar.march_avg_daily_clicks < feb.feb_avg_daily_clicks
            THEN 1
            ELSE 0
        END AS declined_next_month

    FROM february AS feb

    INNER JOIN march AS mar
        ON feb.client_hash_id = mar.client_hash_id
       AND feb.content_hash_id = mar.content_hash_id

    WHERE feb.feb_gsc_days > 0
      AND feb.feb_ga4_days > 0
      AND mar.march_gsc_days > 0
      AND feb.feb_avg_daily_impressions IS NOT NULL
      AND feb.feb_avg_daily_clicks IS NOT NULL
      AND feb.feb_avg_position IS NOT NULL
      AND feb.feb_avg_daily_sessions IS NOT NULL
      AND feb.feb_engagement_rate IS NOT NULL
      AND mar.march_avg_daily_clicks IS NOT NULL
    """
).df()

print("Modelling rows:", len(model_frame))
print("\nLabel counts:")
print(model_frame["declined_next_month"].value_counts())

print("\nLabel percentages:")
print(
    model_frame["declined_next_month"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(model_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modelling rows: 23742

Label counts:
declined_next_month
0    14391
1     9351
Name: count, dtype: int64

Label percentages:
declined_next_month
0    60.61
1    39.39
Name: proportion, dtype: float64


,client_hash_id,content_hash_id,decision_month,feb_avg_daily_impressions,feb_avg_daily_clicks,feb_avg_position,feb_avg_daily_sessions,feb_engagement_rate,march_avg_daily_clicks,declined_next_month
0,client_e547b89c05043229,content_516b7c0e8eec0cef,2026-02,40.678571,0.000000,49.698859,1.000000,0.0,0.000000,0
1,client_e547b89c05043229,content_4e48bd81bb37eb4f,2026-02,262.250000,0.107143,45.407054,1.333333,0.0,0.172414,0
2,client_e547b89c05043229,content_d801c9cbea694166,2026-02,28.535714,0.000000,50.062578,1.000000,0.0,0.068966,0
3,client_e547b89c05043229,content_7e131483384291cf,2026-02,221.642857,0.214286,8.934096,1.625000,0.0,0.137931,1
4,client_e547b89c05043229,content_ecc4400d4e4e5960,2026-02,51.214286,0.000000,14.767085,1.000000,0.0,0.068966,0


### Why each feature was available at decision time

1. **February average daily impressions** was available at decision time
   because it was calculated only from GSC records observed during February.

2. **February average daily clicks** was available at decision time because
   it was calculated only from clicks recorded during February.

3. **February average search position** was available at decision time because
   it used only February search-position and impression information.

4. **February average daily sessions** was available at decision time because
   it was calculated only from GA4 sessions recorded during February.

5. **February engagement rate** was available at decision time because both
   engaged sessions and total sessions had already been measured during
   February.

The March average daily clicks value is future information. It is used only to
create `declined_next_month` and is excluded from the honest feature set.

### Leakage experiment

I first train an honest model using only the five February features.

I then deliberately create a bad feature using March information. Because March
occurs after the February decision point and is used to create the label, this
feature leaks the answer into the model.

The leaked result is shown only as an experiment. The leaked feature is removed
afterward, and only the honest result is retained.

In [15]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier


honest_features = [
    "feb_avg_daily_impressions",
    "feb_avg_daily_clicks",
    "feb_avg_position",
    "feb_avg_daily_sessions",
    "feb_engagement_rate",
]

label = "declined_next_month"

# Do not list feb_avg_daily_clicks separately because it is already
# included inside honest_features.
required_columns = [
    "client_hash_id",
    "march_avg_daily_clicks",
    label,
    *honest_features,
]

# Extra safety: remove any accidental duplicate column names.
required_columns = list(dict.fromkeys(required_columns))

experiment_df = (
    model_frame.loc[:, required_columns]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
    .copy()
)

# Confirm that every column name is unique.
assert experiment_df.columns.is_unique

if experiment_df.empty:
    raise ValueError("No complete rows are available for the experiment.")

if experiment_df[label].nunique() != 2:
    raise ValueError(
        "The label must contain both classes, 0 and 1, before training."
    )

print("Rows used:", len(experiment_df))
print("Columns:", experiment_df.columns.tolist())

print("\nDuplicate columns:")
print(experiment_df.columns[experiment_df.columns.duplicated()].tolist())

print("\nLabel distribution:")
print(experiment_df[label].value_counts())

Rows used: 23742
Columns: ['client_hash_id', 'march_avg_daily_clicks', 'declined_next_month', 'feb_avg_daily_impressions', 'feb_avg_daily_clicks', 'feb_avg_position', 'feb_avg_daily_sessions', 'feb_engagement_rate']

Duplicate columns:
[]

Label distribution:
declined_next_month
0    14391
1     9351
Name: count, dtype: int64


In [16]:
# Hold out complete clients instead of randomly mixing their pages.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_index, test_index = next(
    splitter.split(
        experiment_df[honest_features],
        experiment_df[label],
        groups=experiment_df["client_hash_id"],
    )
)

train_df = experiment_df.iloc[train_index].copy()
test_df = experiment_df.iloc[test_index].copy()

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Clients appearing in both sets:", len(train_clients & test_clients))

print("\nTraining label counts:")
print(train_df[label].value_counts())

print("\nTesting label counts:")
print(test_df[label].value_counts())

Training rows: 22058
Testing rows: 1684
Training clients: 14
Testing clients: 5
Clients appearing in both sets: 0

Training label counts:
declined_next_month
0    13096
1     8962
Name: count, dtype: int64

Testing label counts:
declined_next_month
0    1295
1     389
Name: count, dtype: int64


In [17]:
def evaluate_model(model_name, model, features):
    """Train a model and return test-set metrics."""

    model.fit(train_df[features], train_df[label])
    predictions = model.predict(test_df[features])

    return {
        "model": model_name,
        "accuracy": accuracy_score(test_df[label], predictions),
        "balanced_accuracy": balanced_accuracy_score(
            test_df[label],
            predictions,
        ),
    }


# Baseline that always predicts the most frequent training class.
dummy_model = DummyClassifier(strategy="most_frequent")

# Small readable decision tree using only honest February features.
honest_tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=42,
)

baseline_result = evaluate_model(
    "Dummy baseline",
    dummy_model,
    honest_features,
)

honest_result = evaluate_model(
    "Honest decision tree",
    honest_tree,
    honest_features,
)

honest_results = pd.DataFrame(
    [baseline_result, honest_result]
).set_index("model")

display(honest_results.round(3))

,accuracy,balanced_accuracy
model,,
Dummy baseline,0.769,0.50
Honest decision tree,0.923,0.95


In [18]:
train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)

# Deliberately bad feature: it uses the future March outcome.
train_df["leaked_future_click_delta"] = (
    train_df["feb_avg_daily_clicks"]
    - train_df["march_avg_daily_clicks"]
)

test_df["leaked_future_click_delta"] = (
    test_df["feb_avg_daily_clicks"]
    - test_df["march_avg_daily_clicks"]
)

leaky_features = honest_features + ["leaked_future_click_delta"]

leaked_tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=42,
)

leaked_result = evaluate_model(
    "Leaked decision tree",
    leaked_tree,
    leaky_features,
)

comparison_results = pd.DataFrame(
    [
        baseline_result,
        honest_result,
        leaked_result,
    ]
).set_index("model")

display(comparison_results.round(3))

print(
    "Honest balanced accuracy:",
    round(honest_result["balanced_accuracy"], 3),
)

print(
    "Leaked balanced accuracy:",
    round(leaked_result["balanced_accuracy"], 3),
)


,accuracy,balanced_accuracy
model,,
Dummy baseline,0.769,0.50
Honest decision tree,0.923,0.95
Leaked decision tree,1.000,1.00


Honest balanced accuracy: 0.95
Leaked balanced accuracy: 1.0


In [19]:
leaked_importance = (
    pd.Series(
        leaked_tree.feature_importances_,
        index=leaky_features,
        name="importance",
    )
    .sort_values(ascending=False)
    .to_frame()
)

display(leaked_importance)

,importance
leaked_future_click_delta,1.0
feb_avg_daily_impressions,0.0
feb_avg_daily_clicks,0.0
feb_avg_position,0.0
feb_avg_daily_sessions,0.0
feb_engagement_rate,0.0


In [20]:
train_df.drop(
    columns=["leaked_future_click_delta"],
    inplace=True,
)

test_df.drop(
    columns=["leaked_future_click_delta"],
    inplace=True,
)

assert "leaked_future_click_delta" not in train_df.columns
assert "leaked_future_click_delta" not in test_df.columns

final_features = honest_features.copy()

print("Leaked feature removed successfully.")
print("Final honest features:", final_features)
print(
    "Final honest balanced accuracy:",
    round(honest_result["balanced_accuracy"], 3),
)

Leaked feature removed successfully.
Final honest features: ['feb_avg_daily_impressions', 'feb_avg_daily_clicks', 'feb_avg_position', 'feb_avg_daily_sessions', 'feb_engagement_rate']
Final honest balanced accuracy: 0.95


### Leakage finding

The honest decision tree used only information measured during February. The
leaked model also used `leaked_future_click_delta`, which depended on March
clicks.

March clicks were future information at the February decision point and were
also used to create `declined_next_month`. The leaked feature therefore revealed
the answer instead of genuinely predicting it. Its high score was misleading
and must not be reported as real model performance.

I removed the leaked feature and retained the five February features. The honest
decision-tree result is the reportable result.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [21]:
# Measure how evenly modelling rows are distributed across clients.

pages_per_client = (
    model_frame.groupby("client_hash_id")
    .size()
    .sort_values(ascending=False)
)

data_limits_summary = pd.DataFrame(
    {
        "measure": [
            "Clients represented",
            "Total modelling pages",
            "Minimum pages per client",
            "Median pages per client",
            "Maximum pages per client",
            "Share labelled as decline",
        ],
        "value": [
            model_frame["client_hash_id"].nunique(),
            len(model_frame),
            pages_per_client.min(),
            pages_per_client.median(),
            pages_per_client.max(),
            round(
                model_frame["declined_next_month"].mean() * 100,
                2,
            ),
        ],
    }
)

display(data_limits_summary)

print("\nTen clients with the most modelling rows:")
display(
    pages_per_client
    .head(10)
    .rename("modelling_pages")
    .reset_index()
)

,measure,value
0,Clients represented,19.00
1,Total modelling pages,23742.00
2,Minimum pages per client,1.00
3,Median pages per client,143.00
4,Maximum pages per client,10240.00
5,Share labelled as decline,39.39



Ten clients with the most modelling rows:


,client_hash_id,modelling_pages
0,client_23a62021009f63c4,10240
1,client_e547b89c05043229,6478
2,client_9958f0a7ae1df715,1759
3,client_3197e6291363b4db,1606
4,client_ff644d8251367cbb,1147
5,client_2094c6eb080311d5,1131
6,client_b10cb2997d0c7c86,388
7,client_65de48885f4ef01b,313
8,client_f623b01661d4bfe4,230
9,client_c182d11e4862a37d,143


### Data limitations

1. **Uneven client representation:**  
   Clients do not contribute the same number of pages. Some clients have many
   more modelling rows than others, so the model may be influenced more strongly
   by clients with larger or longer-tracked websites.

2. **Complete-data selection:**  
   The modelling frame includes only pages with usable GSC and GA4 information
   during February and usable GSC information during March. Pages with missing
   tracking data are excluded, so the findings apply only to the observed
   complete-data subset and may not represent every page in the warehouse.

3. **The label does not explain the cause:**  
   `declined_next_month` only shows whether average daily clicks were lower in
   March than in February. It cannot tell whether the decline was caused by
   content quality, seasonality, search-demand changes, website changes, search
   algorithm changes, or another external factor.

4. **Short observation window:**  
   The experiment compares only two adjacent months. A one-month decline may be
   temporary and may not represent a long-term content-performance trend.

### Careful interpretation

The model should be treated as a directional decision-support tool that ranks
pages for human review. It does not prove that a page needs updating, and it
does not establish a causal relationship between the February features and the
March decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.